In [ ]:
#Lab 2.1 — “Taming Gradients & Capacity”
#1. Setup (imports, data, seed)
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt, numpy as np, random, time

def set_seed(s=42):
 random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed(s)
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


tfm = transforms.ToTensor()
train_full = datasets.FashionMNIST('data', train=True, download=True, transform=tfm)
test = datasets.FashionMNIST('data', train=False, download=True, transform=tfm)
n_val = 5000
train, val = random_split(train_full, [len(train_full)-n_val, n_val], generator=torch.Generator().manual_seed(42))
train_dl = DataLoader(train, batch_size=256, shuffle=True)
val_dl = DataLoader(val,   batch_size=512)
test_dl  = DataLoader(test,  batch_size=512)

#2. Deep MLP builder (activation & init as knobs)
class DeepMLP(nn.Module):
    def __init__(self, depth=8, width=256, act='relu', he_init=False):
        super().__init__()
        layers = []
        in_dim = 28*28
        for _ in range(depth):
            layers += [nn.Linear(in_dim, width)]
            in_dim = width
            if act=='relu': layers += [nn.ReLU()]
            elif act=='tanh': layers += [nn.Tanh()]
            elif act=='sigmoid': layers += [nn.Sigmoid()]
        self.feat = nn.Sequential(*layers)
        self.out  = nn.Linear(in_dim, 10)
        # initialization
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if he_init and act=='relu':
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                else:
                    nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.feat(x)
        return self.out(x)

#3. Train loop that logs per-layer gradient norms
def train_one_epoch(model, opt, dl):
    model.train(); tot=0; grad_norms=[]
    for x,y in dl:
        x,y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        # collect grad norms by layer (Linear weights only)
        norms = []
        for m in model.modules():
            if isinstance(m, nn.Linear) and m.weight.grad is not None:
                norms.append(m.weight.grad.detach().norm().item())
        grad_norms.append(norms)
        opt.step(); tot += loss.item()*x.size(0)
    return tot/len(dl.dataset), np.array(grad_norms, dtype=object)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); tot=0; correct=0
    for x,y in dl:
        x,y = x.to(device), y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        tot += loss.item()*x.size(0)
        correct += (logits.argmax(1)==y).sum().item()
    return tot/len(dl.dataset), correct/len(dl.dataset)

#4. Experiment A — Sigmoid vs ReLU + He init (vanish/explode)
def run_experiment(act='sigmoid', he=False, epochs=3):
    model = DeepMLP(depth=8, width=256, act=act, he_init=he).to(device)
    opt = optim.SGD(model.parameters(), lr=0.05)
    hist = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'grad':[]}
    for ep in range(epochs):
        tr_loss, grads = train_one_epoch(model, opt, train_dl)
        va_loss, va_acc = evaluate(model, val_dl)
        hist['train_loss'].append(tr_loss); hist['val_loss'].append(va_loss); hist['val_acc'].append(va_acc); hist['grad'].append(grads)
        print(f"{act.upper()} he={he} | ep{ep+1}  train {tr_loss:.3f}  val {va_loss:.3f}  acc {va_acc:.3f}")
    return model, hist

m_sig, h_sig = run_experiment('sigmoid', he=False, epochs=3)
m_relu_he, h_relu_he = run_experiment('relu', he=True, epochs=3)

#5. Plot gradient norms across depth
def plot_grad_profile(grad_hist, title):
    # average grad norm across batches per layer at final epoch
    g = grad_hist[-1]  # last epoch (list of per-batch lists)
    L = max(len(row) for row in g)
    layer_means = [np.mean([row[i] for row in g if len(row)>i]) for i in range(L)]
    plt.figure(); plt.plot(layer_means, marker='o'); plt.title(title); plt.xlabel("Layer"); plt.ylabel("Gradient norm")

plot_grad_profile(h_sig['grad'], "Sigmoid: gradient norms by depth (epoch 3)")
plot_grad_profile(h_relu_he['grad'], "ReLU + He init: gradient norms by depth (epoch 3)")

#6. Experiment B — Underfitting vs Overfitting
 # small capacity → underfit
small = DeepMLP(depth=1, width=64, act='relu', he_init=True).to(device)
opt = optim.Adam(small.parameters(), lr=1e-3)
for ep in range(5):
    tr, _ = train_one_epoch(small, opt, train_dl)
    vl, va = evaluate(small, val_dl)
    print(f"SMALL ep{ep+1}: train {tr:.3f} val {vl:.3f} acc {va:.3f}")
# big capacity on tiny dataset → overfit
tiny_subset, _ = random_split(train, [1000, len(train)-1000], generator=torch.Generator().manual_seed(42))
tiny_dl = DataLoader(tiny_subset, batch_size=256, shuffle=True)
big = DeepMLP(depth=8, width=512, act='relu', he_init=True).to(device)
opt2 = optim.Adam(big.parameters(), lr=1e-3)
for ep in range(10):
    tr,_ = train_one_epoch(big, opt2, tiny_dl)
    vl, va = evaluate(big, val_dl)
    print(f"BIG-on-TINY ep{ep+1}: train {tr:.3f} val {vl:.3f} acc {va:.3f}")
    
#7. Quick fix: L2 weight decay to reduce overfitting
big2 = DeepMLP(depth=8, width=512, act='relu', he_init=True).to(device)
opt3 = optim.Adam(big2.parameters(), lr=1e-3, weight_decay=1e-3)
for ep in range(10):
    tr,_ = train_one_epoch(big2, opt3, tiny_dl)
    vl, va = evaluate(big2, val_dl)
    print(f"BIG+L2 ep{ep+1}: train {tr:.3f} val {vl:.3f} acc {va:.3f}")